<a href="https://colab.research.google.com/github/animeshsahoo5/Customer_Churn_Prediction/blob/main/Customer_Churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Prediction 📉

 **Churn Prediction Model** using Python, Pandas, Scikit-learn, Logistic Regression, Decision Tree, Random Forest, XGBoost, and Gradio.

## Project workflow

**Customer Data → Data Cleaning → ML Models → Churn Prediction**

This notebook tries to load the public Telco Customer Churn dataset. If the dataset cannot be reached, it creates a clearly labelled synthetic demo dataset so the notebook can still run. The synthetic data is only for practice and should not be treated as real business evidence.
.

## 1. Installing libraries

In [1]:
%pip -q install gradio xgboost

Importing libraries

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## 2. Load customer data

The demo dataset contains customer details and whether each customer left the service.(Data base is random)

In [7]:
def make_demo_data(rows=1800, random_state=42):
    rng = np.random.default_rng(random_state)
    contract = rng.choice(['Month-to-month', 'One year', 'Two year'], rows, p=[0.56, 0.24, 0.20])
    internet = rng.choice(['DSL', 'Fiber optic', 'No'], rows, p=[0.35, 0.48, 0.17])
    online_security = []
    tech_support = []
    for service in internet:
        if service == 'No':
            online_security.append('No internet service')
            tech_support.append('No internet service')
        else:
            online_security.append(rng.choice(['Yes', 'No'], p=[0.38, 0.62]))
            tech_support.append(rng.choice(['Yes', 'No'], p=[0.40, 0.60]))
    paperless = rng.choice(['Yes', 'No'], rows, p=[0.60, 0.40])
    payment = rng.choice(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
                         rows, p=[0.34, 0.22, 0.22, 0.22])
    tenure = rng.integers(0, 73, rows)
    monthly = np.clip(rng.normal(75, 25, rows), 18, 120).round(2)
    monthly = np.where(internet == 'No', np.clip(monthly * 0.35, 18, 55), monthly)
    total = np.maximum(0, tenure * monthly + rng.normal(0, 80, rows)).round(2)

    logit = (-1.0
             + 1.0 * (contract == 'Month-to-month')
             - 0.8 * (contract == 'Two year')
             - 0.035 * tenure
             + 0.018 * (monthly - 70)
             + 0.65 * (np.array(online_security) == 'No')
             + 0.60 * (np.array(tech_support) == 'No')
             + 0.50 * (payment == 'Electronic check'))
    probability = 1 / (1 + np.exp(-logit))
    churn = np.where(rng.random(rows) < probability, 'Yes', 'No')

    return pd.DataFrame({
        'tenure': tenure,
        'MonthlyCharges': monthly,
        'TotalCharges': total,
        'Contract': contract,
        'InternetService': internet,
        'OnlineSecurity': online_security,
        'TechSupport': tech_support,
        'PaperlessBilling': paperless,
        'PaymentMethod': payment,
        'Churn': churn
    })


raw_df = make_demo_data()

print('Rows and columns:', raw_df.shape)
raw_df.head()

Rows and columns: (1800, 10)


,tenure,MonthlyCharges,TotalCharges,Contract,InternetService,OnlineSecurity,TechSupport,PaperlessBilling,PaymentMethod,Churn
0,72,63.54,4553.85,One year,Fiber optic,Yes,Yes,Yes,Credit card (automatic),No
1,45,69.13,3024.90,Month-to-month,Fiber optic,No,Yes,Yes,Credit card (automatic),No
2,3,112.15,396.28,Two year,DSL,No,Yes,Yes,Mailed check,No
3,59,54.44,3194.46,One year,Fiber optic,Yes,Yes,Yes,Electronic check,No
4,56,57.01,3141.70,Month-to-month,Fiber optic,Yes,Yes,Yes,Bank transfer (automatic),No


## 3. Clean the data

We keep nine customer features so the input form stays manageable:
- **Numeric:** tenure, monthly charges, total charges
- **Categorical:** contract, internet service, online security, tech support, paperless billing, payment method

Cleaning steps: convert TotalCharges to a numeric value, replace blank or invalid charges with 0 for customers with little or no tenure, and convert the target (Churn) to 0/1.

In [8]:
FEATURES = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'Contract', 'InternetService', 'OnlineSecurity', 'TechSupport',
    'PaperlessBilling', 'PaymentMethod'
]
TARGET = 'Churn'

# Some rows in the public dataset have blank TotalCharges values.
raw_df['TotalCharges'] = pd.to_numeric(raw_df['TotalCharges'], errors='coerce')
raw_df['TotalCharges'] = raw_df['TotalCharges'].fillna(0)

# Keep only the columns needed for this beginner project.
df = raw_df[FEATURES + [TARGET]].copy()

# Remove rows that have no target label, then standardize the target.
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(str).str.strip().map({'Yes': 1, 'No': 0})
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)

# Ensure numeric features contain numbers.
for column in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
    df[column] = df[column].fillna(df[column].median())

# Fill any missing category with a visible placeholder.
for column in FEATURES:
    if df[column].dtype == 'object':
        df[column] = df[column].fillna('Unknown').astype(str)

print('Data source:', data_source)
print('Rows after cleaning:', len(df))
print('Missing values remaining:', int(df[FEATURES + [TARGET]].isna().sum().sum()))
print('Churn counts (0 = No, 1 = Yes):')
print(df[TARGET].value_counts().sort_index())
df.head()

Data source: Synthetic demo data (online dataset unavailable)
Rows after cleaning: 1800
Missing values remaining: 0
Churn counts (0 = No, 1 = Yes):
Churn
0    1272
1     528
Name: count, dtype: int64


,tenure,MonthlyCharges,TotalCharges,Contract,InternetService,OnlineSecurity,TechSupport,PaperlessBilling,PaymentMethod,Churn
0,72,63.54,4553.85,One year,Fiber optic,Yes,Yes,Yes,Credit card (automatic),0
1,45,69.13,3024.90,Month-to-month,Fiber optic,No,Yes,Yes,Credit card (automatic),0
2,3,112.15,396.28,Two year,DSL,No,Yes,Yes,Mailed check,0
3,59,54.44,3194.46,One year,Fiber optic,Yes,Yes,Yes,Electronic check,0
4,56,57.01,3141.70,Month-to-month,Fiber optic,Yes,Yes,Yes,Bank transfer (automatic),0


## 4. Prepare data for machine learning

- Split data into training and testing sets.
- Use **One-Hot Encoding** for text categories.
- Use **StandardScaler** for numeric columns, which helps Logistic Regression.
- Keep cleaning and transformations inside a pipeline to apply them consistently to new customer inputs.

In [9]:
X = df[FEATURES]
y = df[TARGET]

NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_FEATURES = [
    'Contract', 'InternetService', 'OnlineSecurity', 'TechSupport',
    'PaperlessBilling', 'PaymentMethod'
]

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

Training rows: 1440
Testing rows: 360


## 5. Train and compare four ML models

Training Logistic Regression, Decision Tree, Random Forest, and XGBoost using the same train/test split. Because churn datasets can be imbalanced, we compare **F1 score** as well as accuracy, precision, recall, and ROC-AUC. The highest-F1 model is selected for the default prediction form.This ensures that model with most accurate prediction is chosen as per the data base

In [10]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric='logloss', random_state=42
    )
}

trained_models = {}
results = []

for model_name, classifier in models.items():
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])
    model_pipeline.fit(X_train, y_train)
    predictions = model_pipeline.predict(X_test)
    probabilities = model_pipeline.predict_proba(X_test)[:, 1]

    results.append({
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_test, predictions), 3),
        'Precision': round(precision_score(y_test, predictions, zero_division=0), 3),
        'Recall': round(recall_score(y_test, predictions, zero_division=0), 3),
        'F1 Score': round(f1_score(y_test, predictions, zero_division=0), 3),
        'ROC-AUC': round(roc_auc_score(y_test, probabilities), 3)
    })
    trained_models[model_name] = model_pipeline

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False).reset_index(drop=True)
best_model_name = results_df.loc[0, 'Model']
best_model = trained_models[best_model_name]

print('Best model based on test-set F1 score:', best_model_name)
results_df

Best model based on test-set F1 score: Logistic Regression


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.681,0.469,0.651,0.545,0.741
1,Decision Tree,0.639,0.410,0.519,0.458,0.655
2,Random Forest,0.678,0.440,0.349,0.389,0.685
3,XGBoost,0.697,0.477,0.292,0.363,0.714


### How to read the evaluation table

- **Accuracy:** proportion of all predictions that were correct.
- **Precision:** among customers predicted to churn, how many actually churned.
- **Recall:** among customers who churned, how many the model found.
- **F1 Score:** balances precision and recall.
- **ROC-AUC:** measures how well the model separates churn from non-churn across probability thresholds.

Scores are evaluation results on a held-out test set, not a promise of future performance. If synthetic fallback data was used, the scores only describe that generated demo.

## 7. A Simple Gradio interface



In [19]:
import gradio as gr


def predict_customer_churn(model_name, tenure, monthly_charges, total_charges,
                           contract, internet_service, online_security, tech_support,
                           paperless_billing, payment_method):
    model = trained_models[model_name]
    customer = pd.DataFrame([{
        'tenure': tenure,
        'MonthlyCharges': monthly_charges,
        'TotalCharges': total_charges,
        'Contract': contract,
        'InternetService': internet_service,
        'OnlineSecurity': online_security,
        'TechSupport': tech_support,
        'PaperlessBilling': paperless_billing,
        'PaymentMethod': payment_method
    }])[FEATURES]

    churn_probability = float(model.predict_proba(customer)[0, 1])
    predicted_label = 'May Churn' if churn_probability >= 0.5 else 'Unlikely to Churn'
    probability_result = {
        'May Churn': churn_probability,
        'Unlikely to Churn': 1 - churn_probability
    }

    if churn_probability >= 0.65:
        suggestion = 'Suggested action: review the customer experience and consider a retention offer or support follow-up.'
    elif churn_probability >= 0.40:
        suggestion = 'Suggested action: monitor the account and check whether the customer has unresolved service issues.'
    else:
        suggestion = 'Suggested action: continue normal service and customer engagement.'

    details = (
        f'Chosen model: {model_name}\n'
        f'Prediction: {predicted_label}\n'
        f'Estimated churn probability: {churn_probability:.1%}\n\n'
        f'{suggestion}\n\n'
        'Note: This is a demonstration prediction, not a guarantee. Use appropriate business validation before making customer decisions.'
    )
    return probability_result, details

model_options = list(trained_models.keys())

demo = gr.Interface(
    fn=predict_customer_churn,
    inputs=[
        gr.Dropdown(choices=model_options, value=best_model_name, label='ML Model'),
        gr.Slider(minimum=0, maximum=72, value=4, step=1, label='Tenure (months)'),
        gr.Number(value=85.50, label='Monthly Charges'),
        gr.Number(value=342.00, label='Total Charges'),
        gr.Dropdown(choices=['Month-to-month', 'One year', 'Two year'], value='Month-to-month', label='Contract'),
        gr.Dropdown(choices=['DSL', 'Fiber optic', 'No'], value='Fiber optic', label='Internet Service'),
        gr.Dropdown(choices=['Yes', 'No', 'No internet service'], value='No', label='Online Security'),
        gr.Dropdown(choices=['Yes', 'No', 'No internet service'], value='No', label='Tech Support'),
        gr.Dropdown(choices=['Yes', 'No'], value='Yes', label='Paperless Billing'),
        gr.Dropdown(choices=['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
                    value='Electronic check', label='Payment Method')
    ],
    outputs=[
        gr.Label(num_top_classes=2, label='Churn probability'),
        gr.Textbox(lines=6, label='Prediction details')
    ],
    title='Customer Churn Prediction',
    description='Enter customer information and predict the chance of churn. Model default is the model with the best test F1 score in this run.'
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9bc0c04d566c280f41.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. Explanation

- **Problem:** identify customers who may leave a telecom service.
- **Data cleaning:** convert charge columns, handle missing values, select features, and encode the target.
- **Models compared:** Logistic Regression, Decision Tree, Random Forest, and XGBoost.
- **Evaluation:** accuracy, precision, recall, F1 score, and ROC-AUC on a held-out test split.
- **Interface:** Gradio lets a user enter customer information and compare each trained model.

**Limitations:** Use of rendomly generated Dataset